In [1]:
import pandas as pd

events = pd.read_csv("events.csv")
print(events.shape)
events.head()

# pick 5,000 random items that appear in the events
item_ids = events["itemid"].drop_duplicates().sample(n=5000, random_state=42)
item_set = set(item_ids)
print("Items chosen:", len(item_set))

(2756101, 5)
Items chosen: 5000


In [2]:
for part in [1, 2]:
    source = f"item_properties_part{part}.csv"
    output = f"item_properties_part{part}_sample.csv"
    first_chunk = True

    # read 1 million rows at a time so your RAM doesn't fill up
    for chunk in pd.read_csv(source, chunksize=1_000_000, dtype={"value": str}):
        kept = chunk[chunk["itemid"].isin(item_set)]
        kept.to_csv(output, mode="w" if first_chunk else "a",
                    header=first_chunk, index=False)
        first_chunk = False

    print("Created:", output)

Created: item_properties_part1_sample.csv
Created: item_properties_part2_sample.csv


In [3]:
for part in [1, 2]:
    s = pd.read_csv(f"item_properties_part{part}_sample.csv", dtype={"value": str})
    print(f"part{part} sample:", s.shape, "| unique items:", s["itemid"].nunique())
    display(s.head())

part1 sample: (119904, 4) | unique items: 3983


,timestamp,itemid,property,value
0,1437274800000,186518,available,0
1,1431831600000,15589,6,153715 484436
2,1434250800000,51620,888,1069638
3,1431831600000,243120,159,519769
4,1439694000000,434604,2,769062


part2 sample: (100375, 4) | unique items: 3983


,timestamp,itemid,property,value
0,1431226800000,158384,917,910690
1,1432436400000,155222,678,374401
2,1433041200000,181130,917,229497
3,1436670000000,257218,888,n4992.000 924073 193129 1154859
4,1437274800000,202965,available,1


In [4]:
import pandas as pd

events = pd.read_csv("events.csv")

# same 5,000 items as the item properties sample (same random_state = same items)
item_ids = events["itemid"].drop_duplicates().sample(n=5000, random_state=42)
item_set = set(item_ids)

events_sample = events[events["itemid"].isin(item_set)]
events_sample.to_csv("events_sample.csv", index=False)

print("Original events:", events.shape)
print("Sample events:  ", events_sample.shape)

Original events: (2756101, 5)
Sample events:   (55130, 5)


In [5]:
s = pd.read_csv("events_sample.csv")

print("Unique visitors:", s["visitorid"].nunique())
print("Unique items:   ", s["itemid"].nunique())
print()
print("Events by type:")
print(s["event"].value_counts())
print()
print("Time range:")
print(pd.to_datetime(s["timestamp"].min(), unit="ms"), "to",
      pd.to_datetime(s["timestamp"].max(), unit="ms"))
s.head()

Unique visitors: 40831
Unique items:    5000

Events by type:
event
view           53341
addtocart       1321
transaction      468
Name: count, dtype: int64

Time range:
2015-05-03 03:01:33.434000 to 2015-09-18 02:53:42.658000


,timestamp,visitorid,event,itemid,transactionid
0,1433224229496,15795,view,44872,NaN
1,1433222143427,1130165,view,45337,NaN
2,1433221092368,685949,view,395002,NaN
3,1433222409338,506289,view,131383,NaN
4,1433223968687,338231,view,141264,NaN
